In [1]:
pip install sumy


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean_data.csv")
df.head(3)

,headlines,text,article_len,summary_len
0,Daman & Diu revokes mandatory Rakshabandhan in...,The Daman and Diu administration on Wednesday ...,364,9
1,Malaika slams user who trolled her for 'divorc...,"From her special numbers to TV?appearances, Bo...",396,10
2,'Virgin' now corrected to 'Unmarried' in IGIMS...,The Indira Gandhi Institute of Medical Science...,335,8


In [4]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.summarizers.text_rank import TextRankSummarizer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kartik\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Kartik\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [9]:
def textrank_summarize(text, num_sentences=2):
    parser = PlaintextParser.from_string(text, Tokenizer('english'))
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, num_sentences)
    return " ".join([str(sentence) for sentence in summary])

In [10]:
sample_text = df['text'][0]
sample_headline = df['headlines'][0]

print('original text: ', sample_text)
print('\nactual headline: ', sample_headline)
print('\ntextrank summary: ', textrank_summarize(sample_text))

original text:  The Daman and Diu administration on Wednesday withdrew a circular that asked women staff to tie rakhis on male colleagues after the order triggered a backlash from employees and was ripped apart on social media.The union territory?s administration was forced to retreat within 24 hours of issuing the circular that made it compulsory for its staff to celebrate Rakshabandhan at workplace.?It has been decided to celebrate the festival of Rakshabandhan on August 7. In this connection, all offices/ departments shall remain open and celebrate the festival collectively at a suitable time wherein all the lady staff shall tie rakhis to their colleagues,? the order, issued on August 1 by Gurpreet Singh, deputy secretary (personnel), had said.To ensure that no one skipped office, an attendance report was to be sent to the government the next evening.The two notifications ? one mandating the celebration of Rakshabandhan (left) and the other withdrawing the mandate (right) ? were iss

In [13]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def evaluate_rouge(original_summary, generated_summary):
    scores = scorer.score(original_summary, generated_summary)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure
    }

In [15]:
generated = textrank_summarize(sample_text)
scores = evaluate_rouge(sample_headline, generated)

print("ROUGE-1:", round(scores["rouge1"], 3))
print("ROUGE-2:", round(scores["rouge2"], 3))
print("ROUGE-L:", round(scores["rougeL"], 3))

ROUGE-1: 0.065
ROUGE-2: 0.0
ROUGE-L: 0.065


In [16]:
from tqdm import tqdm

results = []

# run on first 1000 rows only — full dataset will take too long
for i in tqdm(range(1000)):
    try:
        generated = textrank_summarize(df["text"][i])
        scores = evaluate_rouge(df["headlines"][i], generated)
        results.append(scores)
    except:
        pass  # skip rows that cause errors

# calculate averages
avg_rouge1 = sum(r["rouge1"] for r in results) / len(results)
avg_rouge2 = sum(r["rouge2"] for r in results) / len(results)
avg_rougeL = sum(r["rougeL"] for r in results) / len(results)

print(f"Average ROUGE-1: {avg_rouge1:.3f}")
print(f"Average ROUGE-2: {avg_rouge2:.3f}")
print(f"Average ROUGE-L: {avg_rougeL:.3f}")
print(f"\nTotal evaluated: {len(results)} articles")

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [02:22<00:00,  7.00it/s]

Average ROUGE-1: 0.095
Average ROUGE-2: 0.023
Average ROUGE-L: 0.074

Total evaluated: 1000 articles


In [17]:
import json

baseline_scores = {
    "model": "TextRank (extractive)",
    "rouge1": 0.095,
    "rouge2": 0.023,
    "rougeL": 0.074,
    "evaluated_on": "1000 samples"
}

with open("../data/results.json", "w") as f:
    json.dump(baseline_scores, f, indent=4)

print("Baseline scores saved!")

Baseline scores saved!
